### Следующая ячейка - блок импорта общих библиотек и формирования переменных
#### должна выполняться первой

In [1]:
import os # Отключаем попытку tqdm найти визуальный виджет IProgress 
#from tqdm.auto import tqdm
os.environ['TQDM_DISABLE'] = '1'

In [2]:
# ИМПОРТ БИБЛИОТЕК
import os
from collections import Counter
from pathlib import Path
import httpx
from dotenv import load_dotenv
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
# ИМПОРТ ФУНКЦИИ chank_text() вынесенной из учебного примера в /src/ai_engineer_preproject/prj_functions.py
# ФОРМИРОВАНИЕ ПЕРЕМЕННЫХ
# Загружаем переменные из .env
load_dotenv()
# Получаем путь к папке с документами (данные для LLM)
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
DOCS_DIR = Path(os.getenv("OLLAMA_DOCS"))
OLLAMA_URL = os.getenv("OLLAMA_URL")
print(f"Корнева папка проекта:\n{PROJECT_ROOT}")
print(f"Папка с документами датасета:\n{DOCS_DIR}")
print(f"URL для доступа к ollama:\n{OLLAMA_URL}")

Корнева папка проекта:
/home/wirekit69/ai_engineer_preproject
Папка с документами датасета:
/home/wirekit69/ai_engineer_preproject/ollama_docs
URL для доступа к ollama:
http://localhost:11434


In [3]:
# Готовая функция чанкинга
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [4]:
# Проверяем работоспособность сервера ollama и моделей
# Отправляем GET-запрос и получаем ответ
response = httpx.get(f"{OLLAMA_URL}/api/tags")
# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")

Найдено моделей: 4
 - qwen2.5-coder:1.5b
 - qwen2.5-coder:7b
 - lfm3:1.7b
 - qwen2.5:1.5b


In [5]:
for path in sorted(DOCS_DIR.rglob("*"))[:5]:
    print(f"{str(path)} -> {path.name}")


C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api -> api
C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api\anthropic-compatibility.mdx -> anthropic-compatibility.mdx
C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api\authentication.mdx -> authentication.mdx
C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api\balance.mdx -> balance.mdx
C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api\cloud-usage.mdx -> cloud-usage.mdx


In [6]:
# ЗАГРУЖАЕМ ДОКУМЕНТЫ
documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

Загружено документов: 70


In [7]:
# НАРЕЗАЕМ НА ЧАНКИ 
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 609
Средняя длина чанка: 540 символов


In [8]:
# Посмотрим на несколько примеров чанков
for chunk in chunks[200:202]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

--- из C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\capabilities\structured-outputs.mdx (374 символов) ---
response = chat(
  model='gemma4',
  messages=[{
    'role': 'user',
    'content': 'Describe this photo and list the objects you detect.',
    'images': ['path/to/image.jpg'],
  }],
  format=ImageDes ...

--- из C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\capabilities\structured-outputs.mdx (302 символов) ---
## Tips for reliable structured outputs

- Define schemas with Pydantic (Python) or Zod (JavaScript) so they can be reused for validation.
- Lower the temperature (e.g., set it to `0`) for more determ ...



In [9]:
# СВОДКА ПО ЧАНКАМ
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

source_counts = Counter(c["source"] for c in chunks)

top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

Всего чанков: 609
Файлов с чанками: 70
Топ-3 файла по числу чанков:
  C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api.md: 74 чанков
  C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\faq.mdx: 38 чанков
  C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\capabilities\tool-calling.mdx: 33 чанков


In [10]:
file_path = DOCS_DIR / "api.md"
sample_text = file_path.read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    avg_length = sum(len(c) for c in chunks_sample) // len(chunks_sample) if chunks_sample else 0
    print(f"max_chars = {max_chars}\n-> чанков: {len(chunks_sample)}, средняя длина: {avg_length}")


max_chars = 200
-> чанков: 161, средняя длина: 330
max_chars = 800
-> чанков: 74, средняя длина: 736
max_chars = 3000
-> чанков: 30, средняя длина: 1818


In [3]:
# Загрузка модели с сайта HF
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)



Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# ===!!!=== ТОЛЬКО ДЛЯ ЗАГРУЗКИ ЛОКАЛЬНО СОХРАНЕННОЙ МОДЕЛИ ЕМБЕДДИНГОВ ===!!!===
embed_model = SentenceTransformer('local_embed_model')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# ===!!!=== ТОЛЬКО ДЛЯ ЛОКАЛЬНОГО СОХРАНЕНИЯ МОДЕЛИ ЕМБЕДДИНГОВ ===!!!=== 
# folder for save model = {}
# Файловый размер модели около 450 мегабайт
embed_model.save('../saved_models/local_embed_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
# ФОРМИРОВАНИЕ ЕМБЕДДИНГОВ
# # Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Размер матрицы эмбеддингов: (609, 384)


In [ ]:
# ЛОКАЛЬНОЕ СОХРАНЕНИЕ ГОТОВЫХ ЕМБЕДДИНГОВ
np.save('chunk_embeddings.npy',chunk_embeddings)

In [ ]:
# ЗАГРУЗКА ЕМЕБЕДДИНГОВ ИЗ ЛОКАЛЬНО СОХРАНЕННОГО ФАЙЛА
chunk_embeddings = np.load('chunk_embeddings.npy')

In [ ]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

In [ ]:
index = faiss.IndexFlatIP(384)

In [ ]:
dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"В индексе {index.ntotal} векторов размерности {dim}") 

In [ ]:
# ===!!!=== ЛОКАЛЬНОЕ СОХРАНЕНИЕ ИНДЕКСОВ FAISS В ФАЙЛ ===!!!===
index_path = 'MY_index.faiss'  # Имя файла для сохранения индекса
faiss.write_index(index, index_path)

In [ ]:
# ===!!!=== ЗАГРУЗКА ИНДЕКСОВ FAISS ИЗ ФАЙЛА ===!!!===
index_path = 'MY_index.faiss'  # Имя файла для сохранения индекса
index = faiss.read_index(index_path)
#

In [ ]:
# Функция для поиска ближайших чанков по смыслу
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

In [ ]:
question = "Как сменить папку, в которой Ollama хранит модели?"


In [ ]:
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

In [ ]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    results = vector_search(q, top_k=3) 
    for r in results:
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:150].strip()}...\n") 

